# 基于T5的文本摘要

## Step1 导入相关包

In [1]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("./nlpcc_2017/")
ds

Dataset({
    features: ['title', 'content'],
    num_rows: 5000
})

In [3]:
ds = ds.train_test_split(100, seed=42)
ds

DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 4900
    })
    test: Dataset({
        features: ['title', 'content'],
        num_rows: 100
    })
})

In [4]:
ds["train"][0]

{'title': '组图:黑河边防军人零下30℃户外训练,冰霜沾满眉毛和睫毛,防寒服上满是冰霜。',
 'content': '中国军网2014-12-1709:08:0412月16日,黑龙江省军区驻黑河某边防团机动步兵连官兵,冒着-30℃严寒气温进行体能训练,挑战极寒,锻造钢筋铁骨。该连素有“世界冠军的摇篮”之称,曾有5人24人次登上世界军事五项冠军的领奖台。(魏建顺摄)黑龙江省军区驻黑河某边防团机动步兵连官兵冒着-30℃严寒气温进行体能训练驻黑河某边防团机动步兵连官兵严寒中户外训练,防寒服上满是冰霜驻黑河某边防团机动步兵连官兵严寒中户外训练,防寒服上满是冰霜官兵睫毛上都被冻上了冰霜官兵们睫毛上都被冻上了冰霜驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练'}

## Step3 数据处理

In [5]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/mengzi-t5-base")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [6]:
def process_func(exmaples):
    contents = ["摘要生成: \n" + e for e in exmaples["content"]]
    inputs = tokenizer(contents, max_length=384, truncation=True)
    labels = tokenizer(text_target=exmaples["title"], max_length=64, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

In [7]:
tokenized_ds = ds.map(process_func, batched=True)
tokenized_ds

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4900
    })
    test: Dataset({
        features: ['title', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [8]:
tokenizer.decode(tokenized_ds["train"][0]["input_ids"])

'摘要生成: 中国军网2014-12-1709:08:0412月16日,黑龙江省军区驻黑河某边防团机动步兵连官兵,冒着-30°C严寒气温进行体能训练,挑战极寒,锻造钢筋铁骨。该连素有“世界冠军的摇篮”之称,曾有5人24人次登上世界军事五项冠军的领奖台。(魏建顺摄)黑龙江省军区驻黑河某边防团机动步兵连官兵冒着-30°C严寒气温进行体能训练驻黑河某边防团机动步兵连官兵严寒中户外训练,防寒服上满是冰霜驻黑河某边防团机动步兵连官兵严寒中户外训练,防寒服上满是冰霜官兵睫毛上都被冻上了冰霜官兵们睫毛上都被冻上了冰霜驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练驻黑河某边防团机动步兵连官兵严寒中进行户外体能训练</s>'

In [9]:
tokenizer.decode(tokenized_ds["train"][0]["labels"])

'组图:黑河边防军人零下30°C户外训练,冰霜沾满眉毛和睫毛,防寒服上满是冰霜。</s>'

## Step4 创建模型

In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained("../../model/langboat/mengzi-t5-base")

## Step5 创建评估函数

In [11]:
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()

def compute_metric(evalPred):
    predictions, labels = evalPred
    decode_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decode_preds = [" ".join(p) for p in decode_preds]
    decode_labels = [" ".join(l) for l in decode_labels]
    scores = rouge.get_scores(decode_preds, decode_labels, avg=True)
    return {
        "rouge-1": scores["rouge-1"]["f"],
        "rouge-2": scores["rouge-2"]["f"],
        "rouge-l": scores["rouge-l"]["f"],
    }

## Step6 配置训练参数

In [12]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    logging_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="rouge-l",
    predict_with_generate=True
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step7 创建训练器

In [13]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metric,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer)
)

## Step8 模型训练

In [14]:
trainer.train()

  0%|          | 0/459 [00:00<?, ?it/s]

{'loss': 5.6114, 'grad_norm': 5.2806315422058105, 'learning_rate': 4.91285403050109e-05, 'epoch': 0.05}
{'loss': 3.6052, 'grad_norm': 4.610814571380615, 'learning_rate': 4.8257080610021786e-05, 'epoch': 0.1}
{'loss': 3.145, 'grad_norm': 3.4403979778289795, 'learning_rate': 4.738562091503268e-05, 'epoch': 0.16}
{'loss': 3.0058, 'grad_norm': 3.8288190364837646, 'learning_rate': 4.651416122004357e-05, 'epoch': 0.21}
{'loss': 3.0556, 'grad_norm': 3.3829948902130127, 'learning_rate': 4.564270152505447e-05, 'epoch': 0.26}
{'loss': 2.7848, 'grad_norm': 3.406106948852539, 'learning_rate': 4.477124183006536e-05, 'epoch': 0.31}
{'loss': 2.9129, 'grad_norm': 3.956773281097412, 'learning_rate': 4.3899782135076256e-05, 'epoch': 0.37}
{'loss': 2.8338, 'grad_norm': 3.3999650478363037, 'learning_rate': 4.302832244008715e-05, 'epoch': 0.42}
{'loss': 2.715, 'grad_norm': 3.8741414546966553, 'learning_rate': 4.215686274509804e-05, 'epoch': 0.47}
{'loss': 2.781, 'grad_norm': 3.26314115524292, 'learning_rat

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/transformers/generation/utils.py:1249: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 2.1500191688537598, 'eval_rouge-1': 0.47791748050755445, 'eval_rouge-2': 0.30586405509078807, 'eval_rouge-l': 0.4034565711804726, 'eval_runtime': 6.3751, 'eval_samples_per_second': 15.686, 'eval_steps_per_second': 3.922, 'epoch': 1.0}
{'loss': 2.329, 'grad_norm': 2.7452707290649414, 'learning_rate': 3.257080610021787e-05, 'epoch': 1.04}
{'loss': 2.1554, 'grad_norm': 3.577256202697754, 'learning_rate': 3.169934640522876e-05, 'epoch': 1.1}
{'loss': 2.2204, 'grad_norm': 2.957113265991211, 'learning_rate': 3.082788671023965e-05, 'epoch': 1.15}
{'loss': 2.2987, 'grad_norm': 3.3673911094665527, 'learning_rate': 2.9956427015250548e-05, 'epoch': 1.2}
{'loss': 2.273, 'grad_norm': 3.5402255058288574, 'learning_rate': 2.9084967320261443e-05, 'epoch': 1.25}
{'loss': 2.1726, 'grad_norm': 3.227001190185547, 'learning_rate': 2.8213507625272335e-05, 'epoch': 1.31}
{'loss': 2.2132, 'grad_norm': 2.9349663257598877, 'learning_rate': 2.7342047930283227e-05, 'epoch': 1.36}
{'loss': 2.194, 'gr

...

{'eval_loss': 2.0100672245025635, 'eval_rouge-1': 0.4888946139594909, 'eval_rouge-2': 0.32141334558014717, 'eval_rouge-l': 0.41590312018263453, 'eval_runtime': 6.3726, 'eval_samples_per_second': 15.692, 'eval_steps_per_second': 3.923, 'epoch': 2.0}
{'loss': 2.097, 'grad_norm': 2.996624231338501, 'learning_rate': 1.6013071895424836e-05, 'epoch': 2.04}
{'loss': 1.9321, 'grad_norm': 3.003438949584961, 'learning_rate': 1.5141612200435731e-05, 'epoch': 2.09}
{'loss': 1.974, 'grad_norm': 3.087336540222168, 'learning_rate': 1.4270152505446625e-05, 'epoch': 2.14}
{'loss': 2.0047, 'grad_norm': 3.0217912197113037, 'learning_rate': 1.3398692810457516e-05, 'epoch': 2.19}
{'loss': 1.913, 'grad_norm': 2.920766592025757, 'learning_rate': 1.2527233115468408e-05, 'epoch': 2.25}
{'loss': 1.9679, 'grad_norm': 3.008378505706787, 'learning_rate': 1.1655773420479304e-05, 'epoch': 2.3}
{'loss': 1.9043, 'grad_norm': 2.890983819961548, 'learning_rate': 1.0784313725490197e-05, 'epoch': 2.35}
{'loss': 2.0419, 'g

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/transformers/generation/utils.py:1249: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 1.977508544921875, 'eval_rouge-1': 0.4876652852591948, 'eval_rouge-2': 0.3247047753500692, 'eval_rouge-l': 0.4137781625023068, 'eval_runtime': 6.3821, 'eval_samples_per_second': 15.669, 'eval_steps_per_second': 3.917, 'epoch': 3.0}
{'train_runtime': 989.5799, 'train_samples_per_second': 14.855, 'train_steps_per_second': 0.464, 'train_loss': 2.348790590539737, 'epoch': 3.0}


TrainOutput(global_step=459, training_loss=2.348790590539737, metrics={'train_runtime': 989.5799, 'train_samples_per_second': 14.855, 'train_steps_per_second': 0.464, 'total_flos': 5582981834846208.0, 'train_loss': 2.348790590539737, 'epoch': 2.9975510204081632})

## Step9 模型推理

In [15]:
from transformers import pipeline

In [16]:
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

In [17]:
pipe("摘要生成:\n" + ds["test"][-1]["content"], max_length=64, do_sample=True)

[{'generated_text': '中船重工拟定对外投资方案:不涉及公司发行股份,也不涉及公司发行股份或涉及公司股票'}]

In [18]:
ds["test"][-1]["title"]

'中国重工拟以持有的动力相关资产进行对外投资,参与中船重工拟打造的动力业务平台公司,将继续停牌。'